<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/lectures_supplementary_code_samples/l08_seq2seq_attention_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/lectures_supplementary_code_samples/l08_seq2seq_attention_companion.ipynb)

# L08 Companion: Seq2seq, Attention, and Alignment

A decoder needs access to source information at each output step. This workbook traces the move from **one fixed summary** to **query-dependent access**, then distinguishes attention from CTC alignment.

| Section | Guiding question |
|---|---|
| **N1 — Fixed Context** | What must fit into the encoder's final state? |
| **N2 — One Attention Step** | How do scores become a context vector? |
| **N3 — Queries, Keys, and Values** | What chooses access, and what supplies content? |
| **N4 — Attention Matrix** | How do we read rows, columns, and reordering? |
| **N5 — Evidence and Explanation** | What can attention weights tell us about a prediction? |
| **N6 — CTC** | How can frame-level paths produce a shorter label sequence? |

**Predict → run → explain.** Write a prediction before each experiment. Expand solutions after trying the questions. Run setup first, then follow document order; N2–N5 form the main attention route. N1 inspects an architecture without training it. N6 is a separate alignment extension and needs only setup.

<details>
<summary>Setup and notation</summary>

The required packages are `torch` and `matplotlib`; no model downloads are needed. If packages are missing, install them with `%pip install torch matplotlib` in your notebook environment.

Source length is $T$, target length is $M$, and target step is $j$. Mathematical vectors are columns; in code, tokens occupy rows of tensors. All examples use CPU and fixed random seeds. The attention matrix in N4 is hand-crafted, not learned.

</details>

In [ ]:
# Dependencies: PyTorch and matplotlib
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

## N1 — The Fixed-Context Bottleneck

**One final encoder state is the decoder's only source connection in this model.** The decoder also receives previous target tokens, but cannot revisit earlier encoder states.

**Predict:** The hidden width is 32. If the source grows from 5 to 500 tokens, does the final state get wider? Does the number of learned parameters grow?

<details>
<summary>Architecture and scope of the example</summary>

Slide: “A Fixed Context Must Hold the Whole Source.” The encoder processes the source and passes its final state $\mathbf{z}=\mathbf{h}_T$ to the decoder as its initial state $\mathbf{s}_0$. The code inspects this fixed-width connection; it does not train a sequence-reversal model or measure performance versus length.

The state has 32 floating-point coordinates for either source length. That is a fixed representational width, not a claim that it stores exactly 32 tokens or a particular amount of linguistic information. Longer inputs can make it harder to retain all information needed for decoding.

</details>

In [ ]:
class FixedContextSeq2Seq(nn.Module):
    """Plain encoder-decoder: encoder's last hidden state is the bottleneck vector z."""
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.encoder_rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.decoder_rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.out_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src, tgt_in):
        # src: (B, T_src), tgt_in: (B, T_tgt)
        _, z = self.encoder_rnn(self.embed(src))   # z: (1, B, hidden_dim)
        dec_out, _ = self.decoder_rnn(self.embed(tgt_in), z) # bottleneck passed as h_0
        logits = self.out_head(dec_out)            # (B, T_tgt, vocab_size)
        return logits

# Compare parameter count vs input length T
model = FixedContextSeq2Seq(vocab_size=20, emb_dim=32, hidden_dim=32)
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")
print(f"Final-state width: 32 floating-point coordinates per example, for either T=5 or T=500.")

**Check yourself.**

1. For a 500-token source, which encoder state reaches the decoder in this code? Can the decoder directly access $\mathbf{h}_1$?
2. Suppose source words `A B C` roughly align with target words `A′ B′ C′`. If the encoder reads `C B A`, which source word is closest to the first decoding step? What might this help, and what limitation remains?

<details>
<summary>Solution: shorter recurrence paths do not enlarge the summary</summary>

- **The decoder receives only the final state** $\mathbf{h}_{500}$, with 32 features per example. Earlier source information can influence it through recurrence, but the decoder has no direct connection to $\mathbf{h}_1$. The projection and recurrent parameters are reused, so more source tokens do not add parameters or widen the state.
- **Reversing this source brings `A` close to the first output `A′`.** The encoder processes `A` last, just before decoding begins. This shortens the recurrent path for that aligned pair and can make some dependencies easier to learn. It does not shorten every aligned dependency, and the benefit depends on the source–target ordering.
- **The fixed-summary limitation remains.** Reversing the source does not enlarge $\mathbf{z}$ or let the decoder revisit individual encoder states. Attention adds access to those states at each output step; it does not guarantee that every needed detail was retained or retrieved.

</details>

## N2 — One Attention Step

**Scores choose the mixture; values supply its content.** Use encoder states $\mathbf{h}_1=(1,0)^\top$, $\mathbf{h}_2=(0,1)^\top$, and $\mathbf{h}_3=(2,2)^\top$, with scores $(0,2,0)$ for target word *Katze*.

**Predict before running:**

1. Which source position receives the largest weight? Do the two zero-score positions receive zero weight?
2. Compute the normalized weights and the resulting two coordinates of the context vector.

<details>
<summary>Formula and connection to the slide</summary>

Slide: “Your Turn: Compute One Attention Step.” Normalize the scores across source positions, then mix the encoder states:

$$\alpha_{ji}=\frac{\exp(e_{ji})}{\sum_{k=1}^{3}\exp(e_{jk})},\qquad \mathbf{c}_j=\sum_{i=1}^{3}\alpha_{ji}\mathbf{h}_i.$$

The nonnegative weights sum to 1, so the context is a convex combination. A zero score contributes $\exp(0)=1$ to the softmax numerator.

</details>

In [ ]:
# 1. Define states and scores from the slide
H = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [2.0, 2.0]]) # shape: (3, 2) -> (T, d)

scores = torch.tensor([0.0, 2.0, 0.0]) # e_j

# 2. Softmax normalization
weights = F.softmax(scores, dim=0)

# 3. Convex combination (weighted sum)
context = weights @ H # (3,) @ (3, 2) -> (2,)

print("Scores e_j:      ", scores.tolist())
print("Weights alpha_j: ", [round(w, 3) for w in weights.tolist()])
print("Context vector c_j:", [round(c, 3) for c in context.tolist()])

# Verify against analytical numbers from slide: 1/9.4 approx 0.106, 7.4/9.4 approx 0.787
denom = math.exp(0) + math.exp(2) + math.exp(0)
assert math.isclose(weights[1].item(), math.exp(2)/denom, rel_tol=1e-3)

### Equal Scores Versus a Dominant Score

**Predict:** With the same encoder states, compare scores $(1,1,1)$ and $(0,50,0)$.

1. In the equal-score case, what are the weights and the context vector?
2. In the dominant-score case, which encoder state should the context approach?

In [ ]:
# Equal scores (uniform average)
scores_uniform = torch.tensor([1.0, 1.0, 1.0])
weights_uniform = F.softmax(scores_uniform, dim=0)
context_uniform = weights_uniform @ H
print("Uniform context (average of h_i):", context_uniform.tolist())

# Sharp / peaked scores (approaching hard selection)
scores_sharp = torch.tensor([0.0, 50.0, 0.0])
weights_sharp = F.softmax(scores_sharp, dim=0)
context_sharp = weights_sharp @ H
print("Sharp context (approaches h_2):    ", [round(c, 4) for c in context_sharp.tolist()])

**Check yourself.**

1. For scores $(0,2,0)$, what weights and context did you obtain? Why is the context not exactly $\mathbf{h}_2$?
2. How do equal scores and a dominant score change the context?
3. At one decoding step, does the decoder receive all encoder states as separate vectors, or one mixed context vector? How does this differ from one fixed context for the whole output?

<details>
<summary>Solution: one mixture per step, with access to the full source</summary>

- **The weights are approximately $(0.107,0.787,0.107)$.** The context is about $(0.320,1.000)^\top$. Position 2 receives the most weight, but positions 1 and 3 still contribute, so the result is not exactly $\mathbf{h}_2$.
- **Equal scores give the average:** weights $(1/3,1/3,1/3)$ and context $(1,1)^\top$. With scores $(0,50,0)$, almost all weight falls on position 2 and the context approaches $(0,1)^\top$.
- **Each step receives one mixed vector** $\mathbf{c}_j$ in this attention mechanism. All encoder states remain available to the attention computation, which can produce a different mixture for the next query. The decoder no longer has to rely on one unchanged source summary for the entire output. This enables selective access; it does not guarantee perfect retrieval.

</details>

## N3 — Queries, Keys, and Values

**The query and keys determine weights; the values are mixed with those weights.** Here the query comes from the decoder state, and the encoder states serve as both keys and values.

**Predict:** For a batch of 2 examples, 5 source positions, and 24 value features, what are the shapes of the attention weights and the returned context? Can additive attention compare a width-16 query with width-24 keys?

<details>
<summary>Two scoring formulas and their roles</summary>

Slide: “Query, Keys, and Values Name the Three Roles.” In this formulation, $\mathbf{q}_j=\mathbf{s}_{j-1}$ and $\mathbf{k}_i=\mathbf{v}_i=\mathbf{h}_i$.

Additive attention learns a comparison space:

$$e_{ji}=\mathbf{v}_a^\top\tanh(W_q\mathbf{q}_j+W_k\mathbf{k}_i).$$

The parameter $\mathbf{v}_a$ maps comparison features to a scalar score; it is not one of the retrieved value vectors $\mathbf{v}_i$. The two projections allow query and key inputs to have different widths.

A plain dot product instead uses $e_{ji}=\mathbf{q}_j^\top\mathbf{k}_i$, requiring matching query/key widths. The code below implements additive attention; the unscaled dot-product formula is shown for comparison.

</details>

In [ ]:
class BahdanauAdditiveAttention(nn.Module):
    """Bahdanau (additive) attention: score(q, k) = v_a^T tanh(W_q q + W_k k)"""
    def __init__(self, q_dim, k_dim, attn_dim):
        super().__init__()
        self.W_q = nn.Linear(q_dim, attn_dim, bias=False)
        self.W_k = nn.Linear(k_dim, attn_dim, bias=False)
        self.v_a = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, query, keys, values):
        # query:  (B, 1, q_dim)     - decoder state at step j
        # keys:   (B, T, k_dim)     - encoder states
        # values: (B, T, v_dim)     - encoder states
        q_proj = self.W_q(query)    # (B, 1, attn_dim)
        k_proj = self.W_k(keys)     # (B, T, attn_dim)

        # Broadcasting addition across time T
        features = torch.tanh(q_proj + k_proj) # (B, T, attn_dim)
        scores = self.v_a(features).squeeze(-1) # (B, T)

        weights = F.softmax(scores, dim=-1)    # (B, T)
        context = torch.bmm(weights.unsqueeze(1), values) # (B, 1, T) @ (B, T, v_dim) -> (B, 1, v_dim)
        return context.squeeze(1), weights

# Test with a batch of size 2
B, T, q_dim, k_dim, v_dim, attn_dim = 2, 5, 16, 24, 24, 8
attn_module = BahdanauAdditiveAttention(q_dim, k_dim, attn_dim)

q = torch.randn(B, 1, q_dim)
k = torch.randn(B, T, k_dim)
v = k # In standard seq2seq, values are the encoder hidden states themselves

ctx, alpha = attn_module(q, k, v)
print("Context shape: ", ctx.shape)    # (B, v_dim)
print("Weights shape: ", alpha.shape)  # (B, T)
print("Weights sum to 1:", alpha.sum(dim=-1))

**Check yourself.**

1. Check the printed shapes. What does each axis of the weights and context represent?
2. In the call `attn_module(q, k, v)`, `v = k`. How can one encoder state serve two different roles?
3. What flexibility do separate key and value projections give a Transformer?

<details>
<summary>Solution: one representation can serve two computational roles</summary>

- **Weights have shape `(2, 5)`:** one row per batch example and one weight per source position. **Contexts have shape `(2, 24)`:** one mixed vector per example, with the same feature width as the values. The intermediate attention width 8 determines neither the number of source weights nor the context width.
- **Keys are used for scoring; values are used for mixing.** The same encoder state can enter both computations. In the additive scorer, `W_k` already learns a transformation of that state for matching it to the query; the untransformed encoder state supplies the retrieved content in this example.
- **Separate projections let training shape matching and content independently.** A Transformer can compute $\mathbf{k}_i$ and $\mathbf{v}_i$ from the same input using different learned matrices. This separates how a token is matched from what it contributes to the weighted sum. It is additional flexibility, not a requirement that keys and values always differ.

</details>

## N4 — Reading an Attention Matrix

**A row answers: which source positions does this target step access?** Stacking the row distributions gives $A\in\mathbb{R}^{M\times T}$: target steps are rows and source positions are columns.

The example uses *European Economic Area* → *zone économique européenne*. **The weights below are hand-crafted to illustrate reordering; they are not measurements from a trained translation model.**

**Predict:** Which source column should have the largest weight in the row for *zone*? What about *européenne*? Should rows, columns, or both sum to 1?

<details>
<summary>Soft alignment and normalization</summary>

Slide: “The Attention Matrix Shows One Weight Vector per Target Word.” Each target step normalizes over the source positions, so every row sums to 1. A source position may receive attention at several target steps; its column need not sum to 1. A column sum aggregates access weights, not probabilities over mutually exclusive target events.

The phrase reorders approximately from adjective–adjective–noun to noun–adjective–adjective. Soft attention can represent this without requiring monotonic source access.

</details>

In [ ]:
source_tokens = ["[BOS]", "European", "Economic", "Area", "[EOS]"]
target_tokens = ["zone", "économique", "européenne", "[EOS]"]

# Hand-crafted alignment matrix demonstrating non-monotonic reordering
# target rows (M=4) x source cols (T=5)
attn_matrix = torch.tensor([
    [0.02, 0.05, 0.08, 0.82, 0.03],  # zone -> Area
    [0.01, 0.06, 0.85, 0.05, 0.03],  # économique -> Economic
    [0.01, 0.88, 0.05, 0.03, 0.03],  # européenne -> European
    [0.05, 0.02, 0.03, 0.05, 0.85],  # [EOS] -> [EOS]
])

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(attn_matrix.numpy(), cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(len(source_tokens)))
ax.set_yticks(range(len(target_tokens)))
ax.set_xticklabels(source_tokens, fontsize=11)
ax.set_yticklabels(target_tokens, fontsize=11)
ax.set_xlabel("Source tokens (Keys/Values)", fontsize=12, labelpad=10)
ax.set_ylabel("Target tokens (Queries)", fontsize=12)
ax.set_title("Soft Alignment Matrix A [M x T]", fontsize=13, pad=15)

# Annotate cells with values
for i in range(len(target_tokens)):
    for j in range(len(source_tokens)):
        val = attn_matrix[i, j].item()
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=9)

fig.colorbar(cax, fraction=0.046, pad=0.04, label="Attention weight $\\alpha_{ji}$")
plt.tight_layout()
plt.show()

**Check yourself.**

1. Add the values in the row for *zone*. Then add the values in the `[BOS]` column. Why are these sums different?
2. Read the largest weight in each of the three French content-word rows. Which English source positions do they select, and in what order?
3. Does low attention to `[BOS]` establish that this token is unimportant in a real model?

<details>
<summary>Solution: row distributions, accumulated access, and reordering</summary>

- **The `zone` row sums to 1:** $0.02+0.05+0.08+0.82+0.03=1$. Each target step has its own normalized distribution over source positions. The `[BOS]` column sums to $0.02+0.01+0.01+0.05=0.09$; it adds weights from different target steps and is not itself normalized.
- **The strongest links run backward through the content words:** `zone → Area`, `économique → Economic`, and `européenne → European`. The source positions are 3, 2, 1 in the displayed indexing, rather than increasing from left to right. The pattern illustrates noun/adjective reordering.
- **These weights were chosen by hand.** They provide no evidence about a trained model's use of `[BOS]`. Even a measured attention matrix would show one access pattern rather than establish a token's full influence on the prediction.

</details>

## N5 — Attention Weights and Explanations

**A large mixing weight alone does not establish a large effect on the final prediction.** The value being multiplied and the computations after the mixture also matter.

**Predict:** Token 1 has weight 0.95 and value $(0.001,0.001)$. Token 2 has weight 0.05 and value $(10,10)$. Which token contributes more to each coordinate of this weighted sum? Compute both weighted values before running.

<details>
<summary>What this counterexample establishes</summary>

Slide: “Attention Weights Are Not Automatically Explanations.” The example separates a weight from the vector it multiplies. It does not identify a token's causal importance in a trained model. In particular, a zero value has zero direct contribution to the sum, but that position's score can still affect other weights through softmax normalization.

Distinct weight distributions can also produce the same context when the value vectors permit it, such as when two values are identical. Identical contexts then produce identical outputs only if all other inputs to the downstream computation are held fixed. This is a separate limitation of reading weights as unique explanations, not a phenomenon established by the code below.

The lecture discusses these issues through Jain & Wallace (2019) and Wiegreffe & Pinter (2019).

</details>

In [ ]:
# Two tokens with values v1, v2
v1 = torch.tensor([0.001, 0.001]) # Token 1: chosen near-zero value; no claim about a real token type
v2 = torch.tensor([10.0, 10.0])   # Token 2: chosen large value

# Scenario A: Attention is heavily focused on Token 1
alpha_A = torch.tensor([0.95, 0.05])
context_A = alpha_A[0] * v1 + alpha_A[1] * v2

print("Attention weights:", alpha_A.tolist())
print("Contribution of Token 1:", (alpha_A[0] * v1).tolist())
print("Contribution of Token 2:", (alpha_A[1] * v2).tolist())
print("Total Context c_j:      ", context_A.tolist())
print(f"Despite 95% attention on Token 1, Token 2 supplies {((alpha_A[1]*v2[0]) / context_A[0]).item():.1%} of each context coordinate in this example!")

**Check yourself.**

1. Which token contributes more to each coordinate of the context in this example? Why does the largest attention weight give the wrong answer to that question?
2. Does this weighted-sum calculation establish which token determines a trained model's prediction?
3. What additional evidence could gradients or a token-removal experiment provide, and what would remain uncertain?

<details>
<summary>Solution: direct contributions, sensitivity, and interventions</summary>

- **Token 2 contributes more to both coordinates:** $0.05\times10=0.5$, compared with $0.95\times0.001=0.00095$ from token 1. In this positive-valued example, token 2 supplies about 99.8% of each context coordinate. This percentage describes this particular sum, not a general measure of feature importance.
- **The calculation stops at the context vector.** It does not include a trained model's output head or other computational paths. Value directions, cancellation, and later transformations can all affect how the context influences a prediction.
- **Gradients measure local sensitivity** of a chosen output score to an input representation, such as its embedding. They show how that score responds to small changes near the current input; they do not by themselves establish causal importance.
- **Token removal is an intervention.** Compare the chosen output score before and after removing or replacing a token. A change shows an effect under that intervention, which may also alter positions or create an unusual input. No change does not prove that a token is always irrelevant: other tokens or paths may supply redundant information. Use these methods as additional evidence, not as automatic proof.

</details>

## N6 — CTC: Frame-Level Paths to Output Labels

**CTC can train from an input sequence and its label sequence without a known frame-to-label alignment.** A frame-level path assigns one label or blank to each input step. Multiple paths can produce the same shorter output.

**Predict:** What do `h-e-l-l-o` and `h-e-ll-o` become after CTC collapse? Does a blank between two `l` labels make a difference?

<details>
<summary>The two collapse operations and their order</summary>

Slides: “The CTC Core Trick: Blanks and Collapsing” and “The Loss Sums Over All Valid Alignments.” Speech or handwriting inputs are represented as discrete time steps, often more numerous than output labels. Their alignment to the target labels is unknown.

CTC adds a blank symbol, written `-` here. To collapse a path:

1. Merge consecutive identical symbols in the original path.
2. Remove blanks.

The order matters: a blank separates two runs of the same label, allowing both labels to survive. CTC preserves label order; unlike the translation attention example, its paths do not reorder labels.

</details>

In [ ]:
def ctc_collapse(path, blank="-"):
    """Collapsing operator B: merge consecutive duplicates, then drop blanks."""
    # Step 1: collapse consecutive identical tokens
    merged = []
    for token in path:
        if len(merged) == 0 or token != merged[-1]:
            merged.append(token)
    # Step 2: remove blanks
    collapsed = [token for token in merged if token != blank]
    return "".join(collapsed)

# Test cases from the quiz and slides
paths = [
    "c-a-t",       # blanks separating characters
    "cc-aa-tt",    # repeats merged, blanks dropped
    "c-c-a-t",     # blank allows emitting repeated real character 'c'
    "--c-aa-t--",  # leading and trailing blanks
    "h-e-l-l-o",   # 'l' separated by blank -> keeps both 'l's
    "h-e-ll-o"     # consecutive 'l' without blank -> collapses to single 'l'
]

for p in paths:
    print(f"Path: {p:<12} -> Collapsed output: {ctc_collapse(p)}")

### Computing CTC Loss

**The loss includes all paths that collapse to the target, not just one chosen alignment.** The next cell supplies random frame scores for the target `cat`; it demonstrates the loss interface, not a trained recognizer.

**Predict:** Must we supply an exact frame alignment for `cat`, or only its label IDs and length? Why must the frame probabilities include a blank class?

<details>
<summary>Shapes, summation, and loss reduction</summary>

`torch.nn.CTCLoss` takes log probabilities with shape `(T, N, C)`: time steps, batch examples, and classes including blank. The target contains label IDs without blanks. Input and target lengths identify the valid parts of each sequence.

Dynamic programming sums the probabilities of all frame-level paths that collapse to the target, then applies the negative logarithm. The supplied default `reduction="mean"` additionally divides each example's loss by its target length before averaging across examples; the printed scalar is therefore a normalized loss, not the unnormalized negative log probability.

For this example, there are 8 input steps and 3 target labels. Adjacent repeated target labels would require extra input steps to accommodate separating blanks.

</details>

In [ ]:
# T=8 time frames, Batch=1, Num_classes=5 (0: blank, 1: 'c', 2: 'a', 3: 't', 4: other)
T_input = 8
num_classes = 5
blank_idx = 0

# Random logits over 8 audio frames
torch.manual_seed(1)
log_probs = F.log_softmax(torch.randn(T_input, 1, num_classes), dim=-1) # (T, N, C)

# Target sequence: 'c', 'a', 't' -> [1, 2, 3]
targets = torch.tensor([[1, 2, 3]], dtype=torch.long) # (N, S)
input_lengths = torch.tensor([T_input], dtype=torch.long)
target_lengths = torch.tensor([3], dtype=torch.long)

ctc_loss_fn = nn.CTCLoss(blank=blank_idx)
loss = ctc_loss_fn(log_probs, targets, input_lengths, target_lengths)
print(f"CTC Loss over all valid paths: {loss.item():.4f}")

**Check yourself.**

1. Why does `h-e-l-l-o` collapse to `hello`, while `h-e-ll-o` collapses to `helo`?
2. Could a blank-free path produce adjacent repeated output labels under the same collapse rule?
3. Does a CTC blank necessarily mean silence? Does the loss require us to choose one frame alignment for `cat`?

<details>
<summary>Solution: blanks separate label runs; the loss sums valid paths</summary>

- **Merge runs before removing blanks.** In `h-e-l-l-o`, the two `l` runs are separated by a blank and both survive. In `h-e-ll-o`, the adjacent `ll` is one run, so it merges into a single `l`.
- **A separating blank makes adjacent repeated output labels possible.** Without blanks, any adjacent occurrences of the same label merge, while inserting a different label would leave that extra label in the output. For example, `ll-ll` collapses to `ll`, but `llll` collapses to `l`.
- **Blank means “emit no output label at this step.”** It need not correspond to silence or a physical pause between repeated letters. The model can use blanks within continuing speech or handwriting.
- **The target gives the label sequence, not a frame alignment.** CTC loss sums over every valid path that collapses to `cat`. We supply target IDs and lengths, while dynamic programming handles the alternative paths. At inference, simply collapsing the most likely frame-by-frame path is one decoding strategy; it need not yield the most probable output sequence after summing paths.

</details>

## Put the Mechanisms Together

1. What access does attention add that the fixed-context decoder lacks?
2. Why does one attention row sum to 1 while a source column need not?
3. Which mechanism can represent source–target reordering: translation attention or CTC collapse?

<details>
<summary>Check your synthesis</summary>

1. **Access:** Attention can form a new mixture of stored encoder states for each target query. The fixed-context decoder receives only the encoder's final state as its source connection.
2. **Normalization:** Each row is a separate distribution over source positions for one target step. A column adds weights from several such distributions and has no requirement to sum to 1.
3. **Alignment:** Translation attention can access source positions in a different order from the target sequence. CTC collapses monotonic frame-level paths into labels while preserving their order.

</details>